In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import open_clip  # <--- THE FIX
from peft import LoraConfig, get_peft_model
from PIL import Image
import cv2
import numpy as np
import os
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm import tqdm
import torchvision.transforms as T


# ==========================================
# 1. CONFIGURATION
# ==========================================
# OpenCLIP Model Name & Pretrained Tag
# This maps to 'laion/CLIP-convnext_base_w-laion2B-s13B-b82k-augreg' on HF
MODEL_NAME = "convnext_base_w" 
PRETRAINED = "laion2b_s13b_b82k_augreg"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 1  
TILE_SIZE = 512 # ConvNeXt standard resolution
VAL_SPLIT = 0.2
GRAD_ACC = 8

# ==========================================
# 2. IMAGE TILING UTILITY
# ==========================================
def slice_image(image, tile_size=256):
    w, h = image.size
    tiles = []
    for i in range(0, h, tile_size):
        for j in range(0, w, tile_size):
            box = (j, i, min(j + tile_size, w), min(i + tile_size, h))
            tile = image.crop(box)
            if tile.size != (tile_size, tile_size):
                new_tile = Image.new("RGB", (tile_size, tile_size), (0, 0, 0))
                new_tile.paste(tile, (0, 0))
                tile = new_tile
            tiles.append(tile)
    return tiles

# ==========================================
# 3. DATASET (Using OpenCLIP Transforms)
# ==========================================
class TiledTabularDataset(Dataset):
    def __init__(self, preprocess, tokenizer, df, img_dir, transform=None,tile_size=256):
        self.preprocess = preprocess
        self.tokenizer = tokenizer
        self.img_dir = img_dir
        self.df = df
        self.transform = transform
        self.paths = df['image_path'].values
        self.tile_size = tile_size

        # --- NEW: Generate Text Descriptions ---
        # We pre-compute these to avoid string formatting overhead during training
        self.texts = [self._generate_text_description(row) for _, row in df.iterrows()]

    def _generate_text_description(self, row):
        """
        Converts a dataframe row into a descriptive sentence.
        Adjust the template below to emphasize the features you care about most.
        """
        # Option A: Natural Language (Best for CLIP semantics)
        # We focus on the most visually distinct features: Species, State, and key measurements.
        template = (
            f"A photo of {row['Species']} vegetation located in {row['State']}. "
            f"Measurements: Height {row['Height_Ave_cm']:.1f}cm, "
            f"Green Mass {row['Dry_Green_g']:.1f}g, "
            f"Dead Mass {row['Dry_Dead_g']:.1f}g, "
            f"Clover {row['Dry_Clover_g']:.1f}g, "
            f"NDVI {row['Pre_GSHH_NDVI']:.2f}."
        )
        
        # Option B: Key-Value style (Sometimes works better for pure regression tasks)
        # template = (
        #     f"Species: {row['Species']}, State: {row['State']}, "
        #     f"Height: {row['Height_Ave_cm']}, Green: {row['Dry_Green_g']}, "
        #     f"Dead: {row['Dry_Dead_g']}, NDVI: {row['Pre_GSHH_NDVI']}"
        # )
        
        return template

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # 1. Image Loading
        path = os.path.join(self.img_dir, os.path.basename(self.paths[idx]))
        
        # Use OpenCV for speed, but convert to PIL for OpenCLIP compatibility
        img = cv2.imread(path)
        if img is None:
            # Black placeholder if image missing (prevents crash)
            img = np.zeros((1000, 2000, 3), dtype=np.uint8)
        
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        pil_img = Image.fromarray(img) # slice_image expects PIL
        
        if self.transform:
            pil_img = self.transform(pil_img)

        # 2. Tiling
        tiles = slice_image(pil_img, tile_size=self.tile_size)
        
        # 3. Preprocess (Normalization + Tensor conversion)
        # Result: [Num_Tiles, 3, 256, 256]
        tile_tensors = [self.preprocess(tile) for tile in tiles]
        pixel_values = torch.stack(tile_tensors)
        
        return {
            "pixel_values": pixel_values,
            "text": self.texts[idx], # Returns the pre-computed string
            "index": idx
        }

# ==========================================
# 4. MODEL SETUP (OpenCLIP + PEFT)
# ==========================================
def get_lora_model():
    print(f"Loading OpenCLIP model: {MODEL_NAME}...")
    
    # 1. Load Model via OpenCLIP
    model, _, preprocess = open_clip.create_model_and_transforms(
        MODEL_NAME, 
        pretrained=PRETRAINED,
        device=DEVICE
    )
    tokenizer = open_clip.get_tokenizer(MODEL_NAME)
    
    # 2. Enable Checkpointing (Critical for VRAM)
    # OpenCLIP models are wrapped timm models, so we check for the method
    # if hasattr(model.visual, 'set_grad_checkpointing'):
    #     model.visual.set_grad_checkpointing(True)
    # else:
    #     print("Warning: Could not enable gradient checkpointing on visual model.")

    # 3. Freeze Everything
    for param in model.parameters():
        param.requires_grad = False
        
    # 4. Apply LoRA to Visual Encoder ONLY
    # ConvNeXt uses 'fc1', 'fc2' in its MLP blocks
    config = LoraConfig(
        r=8, 
        lora_alpha=16,
        target_modules=["fc1", "fc2"], 
        lora_dropout=0.1,
        bias="none"
    )
    
    # Wrap the visual tower specifically
    model.visual = get_peft_model(model.visual, config)
    
    model.visual.print_trainable_parameters()
    return model, preprocess, tokenizer

def validate(model, loader, text_anchors, device, val_index_offset=0):
    model.eval()
    correct_top1 = 0
    correct_top5 = 0
    total = 0
    total_loss = 0
    loss_fn = nn.CrossEntropyLoss()
    
    with torch.no_grad():
        for i, batch in enumerate(tqdm(loader, desc='val', leave=False)):
            tiles = batch["pixel_values"].squeeze(0).to(device)
            local_index = batch["index"].to(device)
            target_index = local_index + val_index_offset
            
            with torch.amp.autocast('cuda', dtype=torch.float16):
                tile_features = model.encode_image(tiles)
                img_embedding = tile_features.mean(dim=0, keepdim=True)
                img_embedding = img_embedding / img_embedding.norm(dim=-1, keepdim=True)
                
                logit_scale = model.logit_scale.exp()
                logits = (img_embedding @ text_anchors.T) * logit_scale
                
                loss = loss_fn(logits, target_index)
                total_loss += loss.item()
                
                # --- CALCULATE ACCURACY ---
                # Get the top 5 scores and their indices
                # top5_indices shape: [1, 5]
                _, top5_indices = logits.topk(5, dim=-1) 
                
                # Check Top 1
                if top5_indices[0, 0] == target_index:
                    correct_top1 += 1
                # Check Top 5 (Is target inside the list of 5?)
                if target_index in top5_indices[0]:
                    correct_top5 += 1
                
                total += 1
                
    acc_1 = (correct_top1 / total) * 100.0
    acc_5 = (correct_top5 / total) * 100.0
    avg_loss = total_loss / total
    
    return avg_loss, acc_1, acc_5


/home/teo/kaggle/csiro/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from train.train import *
if __name__ == "__main__":
    train_aug = T.Compose([
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.5),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    ])
    df_long = pd.read_csv(os.path.join('csiro-biomass', 'train.csv'))
    df_wide = df_long.pivot(index='image_path', columns='target_name', values='target').reset_index()
    df_wide = df_wide[['image_path'] + ['Dry_Green_g','Dry_Dead_g','Dry_Clover_g','GDM_g','Dry_Total_g']]
    aux_cols = ['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm']
    df_aux = df_long[aux_cols].drop_duplicates().reset_index(drop=True)
    df_wide = df_wide.merge(df_aux, on='image_path', how='left')
    df_wide['biomass_bin'] = pd.qcut(df_wide['Dry_Total_g'], q=10, labels=False)
    df_wide['stratify_key'] = df_wide['Species'].astype(str) + "_" + df_wide['State'].astype(str)
    
    # Check for singletons! 
    # Stratified Split crashes if a group has only 1 member (can't split 1 item into 2 sets).
    # We filter out rare groups or assign them to a "misc" group for the split.
    counts = df_wide['stratify_key'].value_counts()
    singletons = counts[counts < 2].index
    
    # Fallback: If a group has only 1 sample, we treat it as part of a generic group 
    # just for the purpose of splitting, so the code doesn't crash.
    split_key = df_wide['stratify_key'].apply(lambda x: 'misc' if x in singletons else x)

    # --- C. PERFORM STRATIFIED SPLIT ---
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SPLIT, random_state=42)
    
    # This returns the INDICES for the split
    train_idx, val_idx = next(splitter.split(df_wide, split_key))
    train_idx = [
    0, 1, 3, 4, 5, 6, 8, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 22,
    25, 26, 27, 28, 30, 31, 32, 33, 35, 38, 40, 41, 42, 46, 48, 49, 50, 51,
    53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 71, 72,
    73, 74, 75, 76, 77, 78, 79, 80, 81, 83, 84, 85, 86, 87, 88, 89, 90, 91,
    92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 104, 105, 106, 107, 108, 109, 111,
    112, 115, 116, 119, 120, 121, 122, 123, 124, 126, 127, 128, 130, 133, 134, 135, 136, 137,
    138, 139, 141, 142, 143, 144, 145, 146, 148, 149, 150, 152, 153, 154, 155, 157, 158, 159,
    160, 161, 163, 164, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 177, 178, 179, 180,
    183, 184, 185, 186, 187, 188, 190, 192, 193, 194, 196, 197, 199, 200, 201, 202, 204, 205,
    206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223,
    224, 225, 226, 227, 228, 229, 232, 233, 235, 236, 237, 239, 240, 241, 242, 243, 244, 246,
    247, 250, 251, 252, 253, 255, 256, 257, 258, 259, 260, 261, 263, 265, 266, 268, 269, 270,
    271, 272, 275, 276, 277, 278, 279, 281, 283, 284, 285, 287, 289, 290, 291, 293, 294, 295,
    296, 298, 300, 301, 302, 303, 304, 305, 306, 307, 309, 310, 311, 312, 314, 315, 318, 319,
    320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337,
    339, 340, 341, 342, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 355
    ]

    val_idx = [
        2, 7, 9, 18, 21, 23, 24, 29, 34, 36, 37, 39, 43, 44, 45, 47, 52, 69,
        70, 82, 103, 110, 113, 114, 117, 118, 125, 129, 131, 132, 140, 147, 151, 156, 162, 165,
        176, 181, 182, 189, 191, 195, 198, 203, 230, 231, 234, 238, 245, 248, 249, 254, 262, 264,
        267, 273, 274, 280, 282, 286, 288, 292, 297, 299, 308, 313, 316, 317, 338, 343, 354, 356
    ]

    print(val_idx)
    # Create the actual sub-dataframes
    train_df = df_wide.iloc[train_idx].reset_index(drop=True)
    val_df = df_wide.iloc[val_idx].reset_index(drop=True)

    model = train_clip(train_df, val_df)
    #Epoch 1 | Train Loss: 0.6451 | Val Loss: 4.7374 | Val Acc 1: 1.39% | Val Acc 5: 9.72%

[2, 7, 9, 18, 21, 23, 24, 29, 34, 36, 37, 39, 43, 44, 45, 47, 52, 69, 70, 82, 103, 110, 113, 114, 117, 118, 125, 129, 131, 132, 140, 147, 151, 156, 162, 165, 176, 181, 182, 189, 191, 195, 198, 203, 230, 231, 234, 238, 245, 248, 249, 254, 262, 264, 267, 273, 274, 280, 282, 286, 288, 292, 297, 299, 308, 313, 316, 317, 338, 343, 354, 356]
Loading OpenCLIP model: convnext_base_w...
trainable params: 721,920 || all params: 88,943,744 || trainable%: 0.8117
Pre-computing text anchors...
Anchors computed. Starting Training...


Epoch 1 | Train Loss: 0.6452 | Val Loss: 5.1775 | Val Acc 1: 4.17% | Val Acc 5: 6.94%
--> New Best Loss! Saving model...


Epoch 2 | Train Loss: 0.5536 | Val Loss: 5.1678 | Val Acc 1: 1.39% | Val Acc 5: 6.94%
--> New Best Loss! Saving model...


Epoch 3 | Train Loss: 0.5121 | Val Loss: 5.8225 | Val Acc 1: 1.39% | Val Acc 5: 2.78%


Epoch 4 | Train Loss: 0.4749 | Val Loss: 4.9228 | Val Acc 1: 2.78% | Val Acc 5: 16.67%
--> New Best Loss! Saving model...


...

Epoch 10 | Train Loss: 0.4127 | Val Loss: 5.2464 | Val Acc 1: 1.39% | Val Acc 5: 6.94%


Epoch 11 | Train Loss: 0.4118 | Val Loss: 6.0452 | Val Acc 1: 0.00% | Val Acc 5: 5.56%


Epoch 12 | Train Loss: 0.3954 | Val Loss: 6.9276 | Val Acc 1: 1.39% | Val Acc 5: 6.94%


Epoch 13 | Train Loss: 0.3870 | Val Loss: 6.2523 | Val Acc 1: 1.39% | Val Acc 5: 12.50%


Epoch 14 | Train Loss: 0.3847 | Val Loss: 5.3275 | Val Acc 1: 4.17% | Val Acc 5: 12.50%
EARLY STOP (no improvement in 10 epochs)


## Compare pretrained backbones

In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import Dataset, DataLoader
from configs.cfg import CFG
from torch.amp import GradScaler
from dataset.biomass_dataset import *
from utils.augs import *
from configs.deterministic import *
from models.models import *
from train.train import *

if __name__ == "__main__":
    set_seed(CFG.SEED,CFG.DETERMINISTIC)
    df_long = pd.read_csv(CFG.TRAIN_CSV)
    df_wide = df_long.pivot(index='image_path', columns='target_name', values='target').reset_index()
    df_wide = df_wide[['image_path'] + CFG.ALL_TARGET_COLS]
    aux_cols = ['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm']
    df_aux = df_long[aux_cols].drop_duplicates().reset_index(drop=True)
    df_wide = df_wide.merge(df_aux, on='image_path', how='left')
    df_wide['biomass_bin'] = pd.qcut(df_wide['Dry_Total_g'], q=10, labels=False)
    df_wide['stratify_key'] = df_wide['Species'].astype(str) + "_" + df_wide['State'].astype(str)
    
    # Check for singletons! 
    # Stratified Split crashes if a group has only 1 member (can't split 1 item into 2 sets).
    # We filter out rare groups or assign them to a "misc" group for the split.
    counts = df_wide['stratify_key'].value_counts()
    singletons = counts[counts < 2].index
    
    # Fallback: If a group has only 1 sample, we treat it as part of a generic group 
    # just for the purpose of splitting, so the code doesn't crash.
    split_key = df_wide['stratify_key'].apply(lambda x: 'misc' if x in singletons else x)

    # --- C. PERFORM STRATIFIED SPLIT ---
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    
    # This returns the INDICES for the split
    train_idx, val_idx = next(splitter.split(df_wide, split_key))
    print(val_idx)
    # Create the actual sub-dataframes
    train_df = df_wide.iloc[train_idx].reset_index(drop=True)
    val_df = df_wide.iloc[val_idx].reset_index(drop=True)
    g=get_generator()
    train_dataset = BiomassDatasetBase(train_df, get_spatial_transforms(), get_photometric_transforms(), CFG.TRAIN_IMAGE_DIR)
    val_dataset = BiomassDatasetBase(val_df, None, get_val_transforms(), CFG.TRAIN_IMAGE_DIR)
    
    print(f"Data Split: {len(train_dataset)} Training | {len(val_dataset)} Validation")
    
    train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=CFG.NUM_WORKERS, worker_init_fn=seed_worker, generator=g)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=CFG.NUM_WORKERS, worker_init_fn=seed_worker, generator=g)

    model = BiomassModelMLP(
            CFG.MODEL_NAME, 
            freeze_backbone=CFG.FREEZE_BACKBONE,
            checkpoint_path=CFG.CHECKPOINT_PATH
        )
    model = model.to(CFG.DEVICE)
    # print(model)
    if CFG.FREEZE_BACKBONE:
        parameters = filter(lambda p: p.requires_grad, model.parameters())
    else:
        parameters = model.parameters()

    optimizer = torch.optim.AdamW(parameters, lr=CFG.LR, weight_decay=CFG.WD)

    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=1e-2, # Start from a very small LR
        end_factor=1.0,
        total_iters=CFG.WARMUP_EPOCHS
    )
    main_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=CFG.EPOCHS - CFG.WARMUP_EPOCHS
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, main_scheduler],
        milestones=[CFG.WARMUP_EPOCHS]
    )
    scaler = GradScaler()
    best_r2 = -np.inf
    patience = 0
    for epoch in range(1, CFG.EPOCHS+1):
        tr_loss = train_epoch_base(model, train_loader, optimizer, scheduler, CFG.DEVICE, scaler)
        val_loss, val_r2 = valid_epoch_base(model, val_loader, CFG.DEVICE)

        print(f'Epoch {epoch:02d} | '
                    f'TrainLoss {tr_loss:.5f} | '
                    f'ValLoss {val_loss:.5f} | '
                    f'ValR² {val_r2:.4f} {"(BEST)" if val_r2 > best_r2 else ""}')
        if val_r2 > best_r2:
            best_r2 = val_r2
            save_path = os.path.join(CFG.MODEL_DIR, f'best_model.pth')
            torch.save(model.module.state_dict() if hasattr(model, 'module') else model.state_dict(), save_path)
            print(f'   → SAVED (R²: {best_r2:.4f})')
            patience = 0
        else:
            patience += 1
            if patience >= CFG.PATIENCE:
                print(f'   → EARLY STOP (no improvement in {CFG.PATIENCE} epochs)')
                break

    del model, train_loader, val_loader, optimizer, main_scheduler
    torch.cuda.empty_cache()

/home/teo/kaggle/csiro/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[ 29 248 351  25 151 255 312 121 279 307  68 181 264 314 236 280 244 356
 136 335 243 208 354  97 342  82  46 160 173 193 285 117  33  55  83 317
 149 233 124  94   3  80  85 343 300 235 218 100 282  36 184   6  48 115
 155 324 217 177 298 340  62 192 119 180  69  60 303 331 174  67  63 355]
Data Split: 285 Training | 72 Validation
Freezing backbone parameters.


train:   0%|          | 0/285 [00:00<?, ?it/s]/home/teo/kaggle/csiro/.venv/lib/python3.10/site-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 01 | TrainLoss 1917.47366 | ValLoss 1997.91403 | ValR² -1.2947 (BEST)
   → SAVED (R²: -1.2947)


Epoch 02 | TrainLoss 916.00813 | ValLoss 413.76854 | ValR² 0.5248 (BEST)
   → SAVED (R²: 0.5248)


...

Epoch 11 | TrainLoss 174.23951 | ValLoss 262.11724 | ValR² 0.6989 


Epoch 12 | TrainLoss 172.14018 | ValLoss 249.36290 | ValR² 0.7136 


Epoch 13 | TrainLoss 164.28873 | ValLoss 240.09813 | ValR² 0.7242 


Epoch 14 | TrainLoss 137.78337 | ValLoss 303.04395 | ValR² 0.6519 


Epoch 15 | TrainLoss 170.51233 | ValLoss 257.76213 | ValR² 0.7039 
   → EARLY STOP (no improvement in 10 epochs)


## Convert Adapters

In [3]:
import torch
import open_clip
from peft import PeftModel
from collections import OrderedDict

# ==========================================
# 1. CONFIGURATION
# ==========================================
# Must match exactly what you used during training

# The folder where your best LoRA weights are saved
LORA_PATH = "adapters/r8" 
MODEL_NAME = "convnext_base_w" 
PRETRAINED = "laion2b_s13b_b82k_augreg"
# The output file name you will load in the other notebook
OUTPUT_FILENAME = "lora_finetuned_convnext_base_r8.pt"

# ==========================================
# 2. MERGE
# ==========================================
print(f"Loading base model: {MODEL_NAME}...")
base_model, _, _ = open_clip.create_model_and_transforms(
    MODEL_NAME, 
    pretrained=PRETRAINED
)

print(f"Loading LoRA adapters from: {LORA_PATH}...")
base_model.visual = PeftModel.from_pretrained(base_model.visual, LORA_PATH)

print("Merging LoRA weights...")
# This gives us the merged visual model (still has OpenCLIP specific names)
merged_visual_model = base_model.visual.merge_and_unload()
raw_state_dict = merged_visual_model.state_dict()
print(merged_visual_model)
# ==========================================
# 3. CLEANING (The Magic Step)
# ==========================================
print("Cleaning state dict keys...")
clean_state_dict = OrderedDict()

for key, value in raw_state_dict.items():
    # 1. Remove 'trunk.' (OpenCLIP puts everything under trunk for ConvNeXt)
    new_key = key.replace("trunk.", "")
    
    # 2. Remove 'visual.' (Just in case)
    new_key = new_key.replace("visual.", "")
    
    # 3. Remove 'module.' (If DDP was used)
    new_key = new_key.replace("module.", "")
    
    # 4. Handle Head discrepancies
    if "head.proj" in new_key:
        print(f"Skipping CLIP projection layer: {new_key}")
        continue  # Don't add this to the new dict
    
    clean_state_dict[new_key] = value

# ==========================================
# 4. SAVE
# ==========================================
print(f"Saving cleaned weights to {OUTPUT_FILENAME}...")
torch.save(clean_state_dict, OUTPUT_FILENAME)

print("Done! The file is now compatible with standard timm loading.")

Loading base model: convnext_base_w...
Loading LoRA adapters from: adapters/r8...
Merging LoRA weights...
TimmModel(
  (trunk): ConvNeXt(
    (stem): Sequential(
      (0): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((128,), eps=1e-06, elementwise_affine=True)
    )
    (stages): Sequential(
      (0): ConvNeXtStage(
        (downsample): Identity()
        (blocks): Sequential(
          (0): ConvNeXtBlock(
            (conv_dw): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=128)
            (norm): LayerNorm((128,), eps=1e-06, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=128, out_features=512, bias=True)
              (act): GELU()
              (drop1): Dropout(p=0.0, inplace=False)
              (norm): Identity()
              (fc2): Linear(in_features=512, out_features=128, bias=True)
              (drop2): Dropout(p=0.0, inplace=False)
            )
            (shortcut): Ident

## Cross Validation Training

In [1]:
import pandas as pd
import gc
from datetime import datetime
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
from configs.cfg import CFG
from torch.amp import GradScaler
from dataset.biomass_dataset import *
from utils.augs import *
from configs.deterministic import *
from models.models import *
from train.train import *
from dataset.preprocess_data import *
from utils.utils import *

if __name__ == "__main__":
    set_seed(CFG.SEED, CFG.DETERMINISTIC)
    df_wide = get_df()
    sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)

    # Store the best R2 score from each fold
    all_fold_best_scores = []

    for fold, (tr_idx, val_idx) in enumerate(sgkf.split(X=df_wide, y=df_wide['biomass_bin'], groups=df_wide['group'])):
        print('\n' + '='*70)
        print(f'   FOLD {fold+1}/{CFG.N_FOLDS}   |   {len(tr_idx)} train / {len(val_idx)} val')
        print('='*70)

        
        # print(tr_idx)
        # print(val_idx)
        tr_df  = df_wide.iloc[tr_idx].reset_index(drop=True)
        val_df = df_wide.iloc[val_idx].reset_index(drop=True)

        model = train_clip(tr_df, val_df)

        model_state_dict = get_clean_timm_state_dict(model)
        del model
        torch.cuda.empty_cache()
        gc.collect()
        best_r2 = train_base(tr_df,val_df,fold,model_state_dict)
        all_fold_best_scores.append(best_r2)

    # Cleanup
    torch.cuda.empty_cache()
    gc.collect()
    mean_cv_score = np.mean(all_fold_best_scores)
    std_cv_score  = np.std(all_fold_best_scores)

    print('\n' + '='*70)
    print('         FINAL CROSS-VALIDATION SCORE')
    print('='*70)
    print(f'Public LB Score: 0.58')
    print(f'Local CV Score: {mean_cv_score:.3f} ± {std_cv_score:.3f}')
    print('\nIndividual Fold Scores:')
    for i, score in enumerate(all_fold_best_scores):
        print(f'  Fold {i+1}: {score:.4f}')

/home/teo/kaggle/csiro/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
# ===============================================================
# CSIRO Image2Biomass – Training (Weighted R² Validation)
# ===============================================================
import os, gc, cv2, numpy as np, pandas as pd
from tqdm import tqdm
import torch, torch.nn as nn, torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torch.nn.parameter import Parameter
import torch.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold
from torch.amp import GradScaler, autocast
import wandb
from datetime import datetime
from sklearn.model_selection import StratifiedShuffleSplit
import random
import open_clip 
# ---------------------------------------------------------------
# 1. CONFIG (memory-safe + R² metric)
# ---------------------------------------------------------------
class CFG:
    BASE_PATH       = 'csiro-biomass'
    TRAIN_CSV       = os.path.join(BASE_PATH, 'train.csv')
    TRAIN_IMAGE_DIR = os.path.join(BASE_PATH, 'train')
    MODEL_DIR       = 'out'
    N_FOLDS         = 5

    MODEL_NAME = 'convnext_base_w'      # safe & matches inference
    CHECKPOINT_PATH = None #'out/clip_original.pth'#'lora_finetuned_convnext_base.pt'
    IMG_SIZE   = 512                  # fits easily
    PRETRAINED = True
    FREEZE_BACKBONE = True
    CLIP_NAME = "laion2b_s13b_b82k_augreg"

    ALPHA_CLIP   = 0.1
    BATCH_SIZE   = 1
    GRAD_ACC     = 8                  # effective batch = 8
    NUM_WORKERS  = 1
    EPOCHS       = 100
    LR           = 1e-4
    WD           = 0.01 #1e-2 convnext
    PATIENCE     = 10
    WARMUP_EPOCHS = 2
    WARMUP_HEAD_EPOCHS = 5

    DETERMINISTIC = True
    SEED = 694

    TARGET_COLS    = ['Dry_Total_g', 'GDM_g', 'Dry_Green_g']
    DERIVED_COLS   = ['Dry_Clover_g', 'Dry_Dead_g']
    ALL_TARGET_COLS = ['Dry_Green_g','Dry_Dead_g','Dry_Clover_g','GDM_g','Dry_Total_g']
    R2_WEIGHTS     = np.array([0.1, 0.1, 0.1, 0.2, 0.5])  # matches metric
    W_SPECIES = 0.25
    W_STATE = 0.25
    W_CONT = 0.5

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device : {CFG.DEVICE}")
print(f"Backbone: {CFG.MODEL_NAME} | Size: {CFG.IMG_SIZE}")

def set_seed(seed=42, deterministic=True):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# ---------------------------------------------------------------
# 2. WEIGHTED R² METRIC (your function)
# ---------------------------------------------------------------
def global_weighted_r2_score(y_true: np.ndarray, y_pred: np.ndarray):
    """
    Computes the globally weighted R² score as described in the evaluation.
    
    y_true, y_pred: shape (N, 5)
    weights: [0.1, 0.1, 0.1, 0.2, 0.5] (from CFG)
    """
    weights_matrix = np.tile(CFG.R2_WEIGHTS, (y_true.shape[0], 1))
    # y_bar_w = (sum(w_j * y_j)) / (sum(w_j))
    weighted_sum = np.sum(weights_matrix * y_true)
    total_weight = np.sum(weights_matrix)
    y_bar_w = weighted_sum / total_weight # This is a single scalar value
    # SS_res = sum(w_j * (y_j - y_pred_j)^2)
    ss_res = np.sum(weights_matrix * (y_true - y_pred) ** 2)
    # SS_tot = sum(w_j * (y_j - y_bar_w)^2)
    ss_tot = np.sum(weights_matrix * (y_true - y_bar_w) ** 2)
    # R²_w = 1 - (SS_res / SS_tot)
    r2_w = 1 - (ss_res / ss_tot)
    return r2_w
# ---------------------------------------------------------------
# 3. AUGMENTATIONS
# ---------------------------------------------------------------
def get_spatial_transforms():
    # These will be applied to BOTH images identically
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        # A.PadIfNeeded(
        #     min_height=CFG.IMG_SIZE, # Set this to 1024
        #     min_width=CFG.IMG_SIZE,  # Set this to 1024
        #     border_mode=cv2.BORDER_REFLECT_101 # Best padding mode
        # ),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
    ], 
    p=1.0,
    additional_targets={'image_right': 'image'},
    seed=CFG.SEED 
    )
def get_photometric_transforms():
    # These will be applied INDEPENDENTLY to each half
    return A.Compose([
        A.ColorJitter(brightness=0.5,contrast=0.5,saturation=0.5,hue=0.0,p=0.5),
        # A.GaussianBlur(blur_limit=(0, 2), p=0.3),
        # A.GaussNoise(std_range=(0,0.1),mean_range=(0,0),p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std =[0.229, 0.224, 0.225]),
        ToTensorV2()
    ], p=1.0, seed=CFG.SEED)

def get_val_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE), # CFG.IMG_SIZE must be 336
        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std =[0.229, 0.224, 0.225]),
        ToTensorV2()
    ], p=1.0)
# ---------------------------------------------------------------
# X. Discriminative learning rate
# ---------------------------------------------------------------
def create_optimizer_groups(model, lr_heads, lr_backbone):
    """
    Splits model parameters into 'backbone' and 'heads' groups.
    'heads' includes everything that is NOT the backbone.
    """
    parameters = [
        # Group 1: Backbone
        {
            "params": [p for n, p in model.named_parameters() if "backbone" in n],
            "lr": lr_backbone,
            "name": "Backbone"
        },
        # Group 2: Heads (everything else)
        {
            "params": [p for n, p in model.named_parameters() if "backbone" not in n],
            "lr": lr_heads,
            "name": "Heads"
        }
    ]
    
    # Check if the 'Heads' group is empty (which would be a bug)
    if not parameters[1]["params"]:
        print("Warning: 'Heads' group has 0 parameters.")
        
    return parameters
# ---------------------------------------------------------------
# 4. DATASET
# ---------------------------------------------------------------
class BiomassDataset(Dataset):
    def __init__(self, df, transform, photometric_transform, img_dir):
        self.df        = df
        self.transform = transform
        self.ph_transform = photometric_transform
        self.img_dir   = img_dir
        self.paths     = df['image_path'].values
        self.labels    = df[CFG.ALL_TARGET_COLS].values.astype(np.float32)
        self.texts = [self._generate_text_description(row) for _, row in df.iterrows()]

    def _generate_text_description(self, row):
        """
        Converts a dataframe row into a descriptive sentence.
        Adjust the template below to emphasize the features you care about most.
        """
        template = (
            f"A photo of {row['Species']} vegetation located in {row['State']}. "
            f"Measurements: Height {row['Height_Ave_cm']:.1f}cm, "
            f"Green Mass {row['Dry_Green_g']:.1f}g, "
            f"Dead Mass {row['Dry_Dead_g']:.1f}g, "
            f"Clover {row['Dry_Clover_g']:.1f}g, "
            f"NDVI {row['Pre_GSHH_NDVI']:.2f}."
        )
        return template

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        path = os.path.join(self.img_dir, os.path.basename(self.paths[idx]))
        img  = cv2.imread(path)
        if img is None:
            img = np.zeros((1000, 2000, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        h, w, _ = img.shape
        mid = w // 2
        left  = img[:, :mid]
        right = img[:, mid:]
        if self.transform:
            transformed = self.transform(image=left, image_right=right)
            left  = transformed['image']
            right = transformed['image_right']

        # 2. Apply PHOTOMETRIC transforms (independently)
        left  = self.ph_transform(image=left)['image']
        right = self.ph_transform(image=right)['image']

        label = torch.from_numpy(self.labels[idx])
        return left, right, label, idx
    

class BiomassModel(nn.Module):
    def __init__(self, clip_model_name, clip_pretrained, n_species, n_states, n_cont_targets, pretrained=True, freeze_backbone=False, checkpoint_path=None):
        super().__init__()
        
        print(f"Loading OpenCLIP: {clip_model_name} | Tag: {clip_pretrained if pretrained else 'None'}")
        
        model, _, _ = open_clip.create_model_and_transforms(
            clip_model_name, 
            pretrained=clip_pretrained if pretrained else None,
            device='cpu'
        )
        
        self.visual = model.visual
        self.logit_scale = model.logit_scale
        
        head_layers = list(self.visual.head.children())
        
        self.clip_proj = head_layers[-1]
        nf = self.clip_proj.in_features  # The input dim to projection = Raw Feature Dim (e.g. 1024)
        
        feature_extraction_head = nn.Sequential(*head_layers[:-1])
        
        self.backbone = nn.Sequential(
            self.visual.trunk,
            feature_extraction_head
        )
        print(self.backbone)
        # Cleanup
        del model

        # 3. Setup Heads
        image_feature_dim = nf * 2
        print(f"Backbone Features: {nf} | MLP Input: {image_feature_dim}")
        
        self.head = nn.Sequential(
            nn.Linear(image_feature_dim, image_feature_dim//2), 
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(image_feature_dim//2, image_feature_dim//4),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )
        self.regressor = nn.Linear(image_feature_dim//4, 3)

        if freeze_backbone:
            self.freeze_backbone()

    def freeze_backbone(self):
        print("Freezing backbone parameters.")
        for param in self.backbone.parameters():
            param.requires_grad = False
        for param in self.clip_proj.parameters():
            param.requires_grad = False
            
    def unfreeze_backbone(self):
        print("Unfreezing backbone parameters.")
        for param in self.backbone.parameters():
            param.requires_grad = True
        for param in self.clip_proj.parameters():
            param.requires_grad = False

    def forward(self, left, right):
        fl = self.backbone(left)
        fr = self.backbone(right)
        
        # Safety flatten (in case OpenCLIP leaves it as BxCx1x1)
        if len(fl.shape) > 2: fl = fl.flatten(1)
        if len(fr.shape) > 2: fr = fr.flatten(1)
            
        image_features = torch.cat([fl, fr], dim=1) 

        fused = self.head(image_features)
        predictions = self.regressor(fused)
        
        clip_l = self.clip_proj(fl)
        clip_r = self.clip_proj(fr)
        
        # Normalize and Average
        clip_l = clip_l / clip_l.norm(dim=-1, keepdim=True)
        clip_r = clip_r / clip_r.norm(dim=-1, keepdim=True)
        
        scene_embedding = (clip_l + clip_r) / 2.0
        scene_embedding = scene_embedding / scene_embedding.norm(dim=-1, keepdim=True)

        p_total, p_gdm, p_green = predictions.split(1, dim=1)
        return (p_total, p_gdm, p_green), scene_embedding
def weighted_biomass_loss(p_total, p_gdm, p_green, labels, use_huber=False):
    """
    Calculates the 5 individual MSE losses and returns their
    weighted sum, perfectly aligning with the R2 metric weights.
    """
    loss_fn = nn.HuberLoss(delta=15.0) if use_huber else nn.MSELoss()
    
    # 1. Calculate the 5 individual MSE losses
    loss_total = loss_fn(p_total.squeeze(), labels[:, 4]) # Corresponds to Dry_Total_g
    loss_gdm   = loss_fn(p_gdm.squeeze(),   labels[:, 3]) # Corresponds to GDM_g
    loss_green = loss_fn(p_green.squeeze(), labels[:, 0]) # Corresponds to Dry_Green_g

    # Calculate derived predictions
    p_clover = torch.clamp(p_gdm - p_green, min=0)
    p_dead   = torch.clamp(p_total - p_gdm, min=0)

    loss_clover = loss_fn(p_clover.squeeze(), labels[:, 2]) # Corresponds to Dry_Clover_g
    loss_dead   = loss_fn(p_dead.squeeze(),   labels[:, 1]) # Corresponds to Dry_Dead_g

    # 2. Get the weights
    weights = CFG.R2_WEIGHTS
    
    # 3. Apply the weights to their corresponding losses
    weighted_loss_sum = (
        loss_green  * weights[0] +
        loss_dead   * weights[1] +
        loss_clover * weights[2] +
        loss_gdm    * weights[3] +
        loss_total  * weights[4]
    )
    
    return weighted_loss_sum

@torch.no_grad()
def valid_epoch(model, loader, device):
    model.eval()
    running_loss = 0.0
    preds = {'total':[], 'gdm':[], 'green':[]}
    all_labels = []

    for l, r, lab, _ in tqdm(loader, desc='valid', leave=False):
        l, r, lab = l.to(device, non_blocking=True), r.to(device, non_blocking=True), lab.to(device, non_blocking=True)
        with autocast('cuda',dtype=torch.bfloat16):
            (p_tot, p_gdm, p_green), _ = model(l, r)

            loss = weighted_biomass_loss(p_tot, p_gdm, p_green, lab)

        running_loss += loss.item() * l.size(0)

        preds['total'].extend(p_tot.cpu().float().numpy().ravel())
        preds['gdm'].extend(p_gdm.cpu().float().numpy().ravel())
        preds['green'].extend(p_green.cpu().float().numpy().ravel())
        all_labels.extend(lab.cpu().float().numpy())

    # Convert to numpy
    pred_total = np.array(preds['total'])
    pred_gdm   = np.array(preds['gdm'])
    pred_green = np.array(preds['green'])
    true_labels = np.stack(all_labels)  # (N, 5)

    # Compute derived
    pred_clover = np.clip(pred_gdm - pred_green, 0, None)
    pred_dead   = np.clip(pred_total - pred_gdm, 0, None)

    # Stack predictions in correct order
    pred_all = np.stack([
        pred_green,      # Dry_Green_g
        pred_dead,       # Dry_Dead_g
        pred_clover,     # Dry_Clover_g
        pred_gdm,        # GDM_g
        pred_total       # Dry_Total_g
    ], axis=1)

    # Compute weighted R²
    weighted_r2 = global_weighted_r2_score(true_labels, pred_all)

    return running_loss / len(loader.dataset), weighted_r2, pred_all, true_labels

def global_clip_loss(image_embeddings, all_text_anchors, global_indices, logit_scale):
    """
    Calculates CLIP loss against the entire dataset of text anchors.
    
    Args:
        image_embeddings: [Batch_Size, Dim] - The projected image features from the model.
        all_text_anchors: [Total_Dataset_Size, Dim] - Pre-computed embeddings for ALL texts.
        global_indices:   [Batch_Size] - The absolute index (0 to N) of each image in the dataset.
        logit_scale:      Scalar - The learnable temperature parameter.
    """
    image_embeddings = image_embeddings / image_embeddings.norm(dim=-1, keepdim=True)
    
    logits = (image_embeddings @ all_text_anchors.T) * logit_scale.exp()
    
    loss = nn.CrossEntropyLoss()(logits, global_indices)
    
    return loss

# ---------------------------------------------------------------
# 8. TRAINING LOOP
# ---------------------------------------------------------------
def train_epoch(model, loader, opt, scheduler, device, scaler, text_anchors, use_clip=False):
    model.train()
    running = 0.0
    running_clip = 0.0
    opt.zero_grad()
    for i, (l, r, lab, idx) in enumerate(tqdm(loader, desc='train', leave=False)):
        l, r, lab = l.to(device, non_blocking=True), r.to(device, non_blocking=True), lab.to(device, non_blocking=True)
        text_anchors = text_anchors.to(device)
        idx = idx.to(device)
        with autocast('cuda',dtype=torch.bfloat16):

            (p_tot, p_gdm, p_green), img_embeds = model(l, r)
            
            loss_reg = weighted_biomass_loss(p_tot, p_gdm, p_green, lab, use_huber=False)
            total_loss = loss_reg

            if use_clip:
                l_clip = global_clip_loss(img_embeds, text_anchors, idx, model.logit_scale)
                
                total_loss += (0.5 * l_clip)
        
        loss = total_loss / CFG.GRAD_ACC
        scaler.scale(loss).backward()
        # loss.backward()
        running += loss.item() * l.size(0) * CFG.GRAD_ACC
        if use_clip:
            running_clip +=l_clip.item() * l.size(0) * CFG.GRAD_ACC
        if (i + 1) % CFG.GRAD_ACC == 0 or (i + 1) == len(loader):
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            # opt.step()
            opt.zero_grad()
    if use_clip:
        print(f"Clip LOSS: {running_clip}") 
    scheduler.step()
    return running / len(loader.dataset)

def precompute_text_embeddings(dataset, clip_model_name, pretrained_tag, device):
    print("Pre-computing Text Anchors...")
    model, _, _ = open_clip.create_model_and_transforms(clip_model_name, pretrained=pretrained_tag)
    tokenizer = open_clip.get_tokenizer(clip_model_name)
    model = model.to(device).eval()
    
    all_texts = dataset.texts
    all_embeddings = []
    
    with torch.no_grad():
        # Batch size for text encoding
        for i in range(0, len(all_texts), 32):
            batch = all_texts[i:i+32]
            tokens = tokenizer(batch).to(device)
            embed = model.encode_text(tokens)
            embed = embed / embed.norm(dim=-1, keepdim=True)
            all_embeddings.append(embed.cpu())
            
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return torch.cat(all_embeddings, dim=0)

/home/teo/kaggle/csiro/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cuda
Backbone: convnext_base_w | Size: 512


In [ ]:
if __name__ == "__main__":
    set_seed(CFG.SEED, CFG.DETERMINISTIC)
    
    # --- 1. DATA LOADING (Same as before) ---
    df_long = pd.read_csv(os.path.join('csiro-biomass', 'train.csv'))
    df_wide = df_long.pivot(index='image_path', columns='target_name', values='target').reset_index()
    df_wide = df_wide[['image_path'] + ['Dry_Green_g','Dry_Dead_g','Dry_Clover_g','GDM_g','Dry_Total_g']]
    aux_cols = ['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm']
    df_aux = df_long[aux_cols].drop_duplicates().reset_index(drop=True)
    df_wide = df_wide.merge(df_aux, on='image_path', how='left')
    df_wide['biomass_bin'] = pd.qcut(df_wide['Dry_Total_g'], q=10, labels=False)
    df_wide['stratify_key'] = df_wide['Species'].astype(str) + "_" + df_wide['State'].astype(str)
    
    counts = df_wide['stratify_key'].value_counts()
    singletons = counts[counts < 2].index
    split_key = df_wide['stratify_key'].apply(lambda x: 'misc' if x in singletons else x)

    splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, val_idx = next(splitter.split(df_wide, split_key))
    
    train_df = df_wide.iloc[train_idx].reset_index(drop=True)
    val_df = df_wide.iloc[val_idx].reset_index(drop=True)

    tr_set = BiomassDataset(train_df, get_spatial_transforms(), get_photometric_transforms(), CFG.TRAIN_IMAGE_DIR)
    val_set= BiomassDataset(val_df, None, get_val_transforms(), CFG.TRAIN_IMAGE_DIR)

    tr_text_anchors = precompute_text_embeddings(tr_set, CFG.MODEL_NAME, CFG.CLIP_NAME, CFG.DEVICE)
    # val_text_anchors = precompute_text_embeddings(val_set, CFG.MODEL_NAME, CFG.CLIP_NAME, CFG.DEVICE)

    def seed_worker(worker_id):
        s = torch.initial_seed() % 2**32
        np.random.seed(s)
        random.seed(s)
    g = torch.Generator()
    g.manual_seed(CFG.SEED)

    tr_loader  = DataLoader(tr_set,  batch_size=CFG.BATCH_SIZE, shuffle=True,
                            num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True, worker_init_fn=seed_worker,generator=g)
    val_loader = DataLoader(val_set, batch_size=CFG.BATCH_SIZE, shuffle=False,
                            num_workers=CFG.NUM_WORKERS, pin_memory=True, worker_init_fn=seed_worker,generator=g)
    
    # --- 2. MODEL INIT ---
    model = BiomassModel(
            CFG.MODEL_NAME, 
            CFG.CLIP_NAME,
            n_species=1,
            n_states=1,
            n_cont_targets=1,
            pretrained=CFG.PRETRAINED, 
            # Note: We manage freezing manually in the loop now, so init with False is fine
            freeze_backbone=False, 
            checkpoint_path=CFG.CHECKPOINT_PATH
        )
    model = model.to(CFG.DEVICE)
    scaler = GradScaler()
    best_r2 = -np.inf
    
    # We define how many epochs to spend just warming up the head
    HEAD_WARMUP_EPOCHS = 2 
    TOTAL_EPOCHS = CFG.EPOCHS

    # ====================================================
    # PHASE 1: HEAD WARMUP (Backbone Frozen)
    # ====================================================
    print(f"\n🔒 PHASE 1: HEAD WARMUP (Epochs 1-{HEAD_WARMUP_EPOCHS})")
    
    # 1. Freeze Backbone
    model.freeze_backbone()
    
    # 2. Optimizer for Head Only
    # We filter only parameters that require grad (which is now only the head)
    head_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(head_params, lr=CFG.LR, weight_decay=CFG.WD)
    
    # 3. Simple Scheduler for Warmup
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=HEAD_WARMUP_EPOCHS)

    for epoch in range(1, HEAD_WARMUP_EPOCHS + 1):
        tr_loss = train_epoch(model, tr_loader, optimizer, scheduler, CFG.DEVICE, scaler, tr_text_anchors)
        val_loss, val_r2, _, _ = valid_epoch(model, val_loader, CFG.DEVICE)

        print(f'Warmup {epoch}/{HEAD_WARMUP_EPOCHS} | Train: {tr_loss:.4f} | Val: {val_loss:.4f} | R²: {val_r2:.4f}')

    del head_params, optimizer, scheduler
    # ====================================================
    # PHASE 2: FULL FINE-TUNING (Backbone Unfrozen)
    # ====================================================
    print(f"\n🔓 PHASE 2: FULL FINE-TUNING (Epochs {HEAD_WARMUP_EPOCHS+1}-{TOTAL_EPOCHS})")
    
    # 1. Unfreeze Backbone
    model.unfreeze_backbone()
    
    # 2. Re-create Optimizer with Discriminative Learning Rates
    # Use your helper function to give backbone 1/100th the LR of the head
    parameters = create_optimizer_groups(
        model=model,
        lr_heads=CFG.LR,
        lr_backbone=CFG.LR / 100 
    )
    optimizer = optim.AdamW(parameters, lr=CFG.LR, weight_decay=CFG.WD)

    # 3. New Scheduler for the rest of training
    # We restart the schedule for the long haul
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=TOTAL_EPOCHS - HEAD_WARMUP_EPOCHS,
        eta_min=1e-7
    )

    patience = 0
    for epoch in range(HEAD_WARMUP_EPOCHS + 1, TOTAL_EPOCHS + 1):
        tr_loss = train_epoch(model, tr_loader, optimizer, scheduler, CFG.DEVICE, scaler,tr_text_anchors, use_clip=True)
        val_loss, val_r2, _, _ = valid_epoch(model, val_loader, CFG.DEVICE)

        print(f'Epoch {epoch:02d} | '
                f'TrainLoss {tr_loss:.5f} | '
                f'ValLoss {val_loss:.5f} | '
                f'ValR² {val_r2:.4f} {"(BEST)" if val_r2 > best_r2 else ""}')
        
        if val_r2 > best_r2:
            best_r2 = val_r2
            save_path = os.path.join(CFG.MODEL_DIR, f'clip_finetuned_best.pth')
            torch.save(model.state_dict(), save_path) # No need for module check if not using DataParallel
            print(f'   → SAVED (R²: {best_r2:.4f})')
            patience = 0
        else:
            patience += 1
            if patience >= CFG.PATIENCE:
                print(f'   → EARLY STOP (no improvement in {CFG.PATIENCE} epochs)')
                break

# Epoch 73 | TrainLoss 156.18394 | ValLoss 243.00194 | ValR² 0.7209 (BEST) 
# Epoch 36 | TrainLoss 181.11488 | ValLoss 267.50694 | ValR² 0.6927 (BEST)

Pre-computing Text Anchors...
Loading OpenCLIP: convnext_base_w | Tag: laion2b_s13b_b82k_augreg
Sequential(
  (0): ConvNeXt(
    (stem): Sequential(
      (0): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((128,), eps=1e-06, elementwise_affine=True)
    )
    (stages): Sequential(
      (0): ConvNeXtStage(
        (downsample): Identity()
        (blocks): Sequential(
          (0): ConvNeXtBlock(
            (conv_dw): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=128)
            (norm): LayerNorm((128,), eps=1e-06, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=128, out_features=512, bias=True)
              (act): GELU()
              (drop1): Dropout(p=0.0, inplace=False)
              (norm): Identity()
              (fc2): Linear(in_features=512, out_features=128, bias=True)
              (drop2): Dropout(p=0.0, inplace=False)
            )
            (shortcut): Identity()
       

train:   0%|          | 0/285 [00:00<?, ?it/s]/home/teo/kaggle/csiro/.venv/lib/python3.10/site-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Warmup 1/2 | Train: 1553.2433 | Val: 1054.7538 | R²: -0.2114


Warmup 2/2 | Train: 765.7432 | Val: 690.7558 | R²: 0.2066

🔓 PHASE 2: FULL FINE-TUNING (Epochs 3-100)
Unfreezing backbone parameters.


Clip LOSS: 13779.625


...

Epoch 44 | TrainLoss 129.22531 | ValLoss 419.99679 | ValR² 0.5176 


Clip LOSS: 12085.5


Epoch 45 | TrainLoss 127.79336 | ValLoss 355.95147 | ValR² 0.5912 


Clip LOSS: 12041.375


Epoch 46 | TrainLoss 138.72534 | ValLoss 297.38750 | ValR² 0.6584 
   → EARLY STOP (no improvement in 10 epochs)
